In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
if os.path.exists('/content/korean-chatbot'):
    %cd /content/korean-chatbot
    !git pull
else:
    %cd /content
    !git clone https://github.com/kkkk2058/korean-chatbot.git
    %cd korean-chatbot

!pip install transformers torch tokenizers
import sys
sys.path.append('/content/korean-chatbot/src')

In [ ]:
import shutil, os

# tokenizer.json 가져오기
os.makedirs("tokenizer", exist_ok=True)
shutil.copy("/content/drive/MyDrive/korean-chatbot/tokenizer.json", "tokenizer/tokenizer.json")

# train.txt 가져오기
os.makedirs("data", exist_ok=True)
shutil.copy("/content/drive/MyDrive/korean-chatbot/train.txt", "data/train.txt")

print("파일 로드 완료!")
print(f"train.txt 크기: {os.path.getsize('data/train.txt') / 1024 / 1024:.1f} MB")

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
from tokenizer import KoreanTokenizer
from model import Transformer

kt = KoreanTokenizer()
kt.load("tokenizer/tokenizer.json")

model = Transformer(vocab_size=kt.vocab_size)
model = model.to(device)
print(f"vocab size: {kt.vocab_size}")
print(f"파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, file_path, tokenizer, max_len=512):
        self.samples = []
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                ids = tokenizer.encode(line.strip())
                if len(ids) > 1:
                    ids = ids[:max_len]
                    self.samples.append(torch.tensor(ids))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def collate_fn(batch):
    return torch.nn.utils.rnn.pad_sequence(batch, batch_first=True, padding_value=0)

dataset = TextDataset("data/train.txt", kt)
loader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
print(f"샘플 수: {len(dataset)}")
print(f"배치 수: {len(loader)}")

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(3):
    total_loss = 0
    for i, batch in enumerate(loader):
        batch = batch.to(device)
        loss = model.loss(batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
        if i % 100 == 0:
            print(f"epoch {epoch+1} | step {i}/{len(loader)} | loss: {loss.item():.4f}")
    
    print(f"✅ epoch {epoch+1} 완료 | avg loss: {total_loss/len(loader):.4f}")

In [ ]:
import shutil

os.makedirs("checkpoints", exist_ok=True)
torch.save(model.state_dict(), "checkpoints/model.pt")

# Drive 백업
os.makedirs("/content/drive/MyDrive/korean-chatbot/checkpoints", exist_ok=True)
shutil.copy("checkpoints/model.pt", "/content/drive/MyDrive/korean-chatbot/checkpoints/model.pt")
print("모델 저장 & Drive 백업 완료!")